In [1]:
# Let's add the root directory to our system search path to allow imports from sibling directories.
import os, sys
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import random
from pathlib import Path

import cv2
import torch
import faiss
import numpy as np
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

from src.utils import load_backbone, run_inference, show_images, print_iou

WEIGHTS_DIR = Path("../weights/")
EPOCH_COUNT = len(list(WEIGHTS_DIR.iterdir())) - 2

DEVICE = torch.device("cuda")  # ConvNeXtV2 doesn't support CPU

SEED = 42
RNG = np.random.default_rng(seed=SEED)
random.seed(SEED)

DATASET_PATH = Path("../data/labelled/s3seg")
IMGS_PATH = DATASET_PATH / "images/"
MASK_PATH = DATASET_PATH / "masks/"
EMBEDS_PATH = DATASET_PATH / "embeds/"

# Few Shot Scalability (1%, 5%, 10%, 50%, 100%)
FEW_SHOT = (0.01, 0.05, 0.10, 0.50, 1.00)

NUM_SAMPLES = 10_000

DATA_EXT = '.tiff'
DATA_FILES = random.sample(sorted([im.stem for im in IMGS_PATH.glob(f"*{DATA_EXT}")]), k=NUM_SAMPLES)

N = len(DATA_FILES)
TEST_SPLIT = int(0.2 * N)

/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/hayat/projects/benthic/venv/lib/python3.8/site-packages/spconv/pytorch/functional.py:243: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Pl

In [3]:
class ApproxKNeighborsClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=10, weights="distance", M=32, eps=1e-8):
        """
        Custom Scikit-Learn wrapper for FAISS Approximate Nearest Neighbors.

        Args:
            n_neighbors (int): Number of neighbors to use.
            weights (str): 'uniform' for majority vote, 'distance' for inverse distance weighting.
            M (int): HNSW graph parameter. Higher means more accurate but slower (default: 32).
        """
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.M = M
        self.eps = eps

    def fit(self, X, y):
        # Validate inputs and store classes
        X, y = check_X_y(X, y)

        self.n_samples_fit_ = X.shape[0]

        # Safely encode labels to contiguous integers (0 to C-1)
        self.le_ = LabelEncoder()
        self.y_train_ = self.le_.fit_transform(y)
        self.classes_ = self.le_.classes_

        X_np = np.ascontiguousarray(X, dtype=np.float32)
        d = X_np.shape[1]

        # Initialize and populate the HNSW index
        self.index_ = faiss.IndexHNSWFlat(d, self.M)
        self.index_.add(X_np)

        return self

    def predict_proba(self, X):
        """Returns class probabilities for each sample."""
        check_is_fitted(self)
        X = check_array(X)
        X_np = np.ascontiguousarray(X, dtype=np.float32)

        k = min(self.n_neighbors, self.n_samples_fit_)

        # Search the index
        distances, indices = self.index_.search(X_np, k)

        # Handle FAISS -1 padding for missing neighbors
        invalid_mask = (indices == -1)
        safe_indices = np.maximum(indices, 0)
        neighbor_labels = self.y_train_[safe_indices]

        N, k_actual = neighbor_labels.shape
        num_classes = len(self.classes_)

        if self.weights == "distance":
            # FAISS returns squared L2. Take sqrt to match sklearn's standard inverse distance.
            safe_distances = np.maximum(distances, 0.0)
            weights = 1.0 / (np.sqrt(safe_distances) + self.eps)
        else:
            # Standard uniform majority vote
            weights = np.ones_like(distances, dtype=np.float32)

        # Zero out the weights of invalid (padded) neighbors
        weights[invalid_mask] = 0.0

        # Vectorized Weighted Voting
        row_offsets = np.arange(N) * num_classes
        flat_labels = (neighbor_labels + row_offsets[:, None]).ravel()
        flat_weights = weights.ravel()

        # Single C-level bincount for the entire dataset
        flat_counts = np.bincount(flat_labels, weights=flat_weights, minlength=N * num_classes)

        # Reshape back to rows (N, num_classes)
        counts_2d = flat_counts.reshape(N, num_classes)

        # Normalize rows to create probabilities
        row_sums = counts_2d.sum(axis=1, keepdims=True)
        # Avoid division by zero if all neighbors were invalid
        row_sums[row_sums == 0.0] = 1.0 

        return counts_2d / row_sums

    def predict(self, X):
        # Predict uses the argmax of predict_proba
        proba = self.predict_proba(X)
        preds = np.argmax(proba, axis=1)
        return self.classes_[preds]

In [4]:
EMBEDS_PATH.mkdir(exist_ok=True)

embed_files = list(EMBEDS_PATH.glob('*.npz'))
if len(embed_files) < N:
    model = load_backbone(WEIGHTS_DIR / "checkpoint_latest.pth")

    for file in tqdm(DATA_FILES):
        embed_path = EMBEDS_PATH / (file + '.npz')
        if embed_path.exists():
            continue

        img_path = IMGS_PATH / (file + DATA_EXT)
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
        img = torch.from_numpy(img[np.newaxis, :, :])  # (W, H) -> (1, W, H)

        cls, flat_patches, outputs = run_inference(model, img)
        np.savez_compressed(
            embed_path,
            cls=cls,
            stage0=outputs[0],
            stage1=outputs[1],
            stage2=outputs[2],
            stage3=outputs[3],
            stage4=outputs[4],  # Fused Hypercolumn
        )

In [5]:
masks, embeds = None, None
for i, file in enumerate(tqdm(DATA_FILES)):
    mask_path = MASK_PATH / (file + DATA_EXT)
    embed_path = EMBEDS_PATH / (file + '.npz')

    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED).astype(np.float32)
    data = np.load(embed_path)

    if masks is None or embeds is None:
        masks = np.zeros([N, ] + list(mask.shape), dtype=np.float32)

        embed = (data['cls'], data['stage0'], data['stage1'], data['stage2'], data['stage3'], data['stage4'])
        embeds = [
            np.zeros([N, ] + list(e.shape), dtype=np.float32)
            for e in embed
        ]

    masks[i] = mask

    embeds[0][i] = data['cls']
    embeds[1][i] = data['stage0']
    embeds[2][i] = data['stage1']
    embeds[3][i] = data['stage2']
    embeds[4][i] = data['stage3']
    embeds[5][i] = data['stage4']

    data.close()
    del mask, data

unique_labels = np.unique(masks)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [6]:
# Label guide: Class 0 -> Sand Ripples, Class 1 -> Fine Sediments, Class 2 -> Rocks, Class 3 -> Maerl

classifiers = []
for stage in range(5):
    X_train = embeds[stage + 1][TEST_SPLIT:]

    _, H_orig, W_orig = masks.shape
    _, H_feat, W_feat, C = X_train.shape

    # Exact Nearest-Neighbor mask downsampling (center-aligned)
    row_idx = ((np.arange(H_feat) + 0.5) * (H_orig / H_feat)).astype(int)
    col_idx = ((np.arange(W_feat) + 0.5) * (W_orig / W_feat)).astype(int)

    y_train = masks[TEST_SPLIT:][:, row_idx[:, None], col_idx]

    X_test = embeds[stage + 1][:TEST_SPLIT].reshape(-1, C)  # (N * H * W, C)
    y_test = masks[:TEST_SPLIT][:, row_idx[:, None], col_idx].reshape(-1)  # (N * H * W)

    for shot in FEW_SHOT:
        num_samples = int(shot * (N - TEST_SPLIT))

        X_samples = X_train[:num_samples].reshape(-1, C)  # (N * H * W, C)
        y_samples = y_train[:num_samples].reshape(-1)  # (N * H * W)

        linear_probe = make_pipeline(
            StandardScaler(),
            SGDClassifier(
                loss="log_loss",          # Logistic regression loss
                penalty="l2",             # L2 (Ridge) regularization
                alpha=1e-4,
                max_iter=1000,
                tol=1e-3,
                early_stopping=True,
                validation_fraction=0.1,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1
            )
        )

        linear_probe.fit(X_samples, y_samples)
        lin_train_preds = linear_probe.predict(X_samples)
        lin_test_preds = linear_probe.predict(X_test)

        print(f"=== Stage {stage + 1} | Few-Shot Scale: {shot * 100:.0f}% ===")

        print("\n-- Linear Probe --")
        print("Train Set:")
        print(classification_report(y_samples, lin_train_preds, digits=3))
        print_iou(y_samples, lin_train_preds, unique_labels)

        print("\nTest Set:")
        print(classification_report(y_test, lin_test_preds, digits=3))
        print_iou(y_test, lin_test_preds, unique_labels)

        knn_probe = make_pipeline(
            StandardScaler(),
            ApproxKNeighborsClassifier(
                n_neighbors=10,  # Based on Two-NN test
                weights="distance",
            )
        )

        knn_probe.fit(X_samples, y_samples)
        knn_train_preds = knn_probe.predict(X_samples)
        knn_test_preds = knn_probe.predict(X_test)

        print("\n-- K-Nearest Neighbors --")
        print("Train Set:")
        print(classification_report(y_samples, knn_train_preds, digits=3))
        print_iou(y_samples, knn_train_preds, unique_labels)

        print("\nTest Set:")
        print(classification_report(y_test, knn_test_preds, digits=3))
        print_iou(y_test, knn_test_preds, unique_labels)

        print("\n" + "="*50 + "\n\n")

        classifiers.append(linear_probe)
        del knn_probe, knn_train_preds, knn_test_preds, lin_train_preds, lin_test_preds

    del X_train, X_test
    del y_train, y_test


=== Stage 1 | Few-Shot Scale: 1% ===

-- Linear Probe --
Train Set:
              precision    recall  f1-score   support

         0.0      0.772     0.798     0.785    157503
         1.0      0.712     0.575     0.636     94427
         2.0      0.268     0.311     0.288     35337
         3.0      0.554     0.652     0.599     40413

    accuracy                          0.663    327680
   macro avg      0.577     0.584     0.577    327680
weighted avg      0.673     0.663     0.666    327680

Confusion Matrix (Normalized):
 True / Pred |     0.0     1.0     2.0     3.0
----------------------------------------------
         0.0 |   0.798   0.080   0.100   0.022
         1.0 |   0.148   0.575   0.110   0.167
         2.0 |   0.550   0.084   0.311   0.055
         3.0 |   0.094   0.159   0.096   0.652
----------------------------------------------

IoU per class:
  Class 0.0  : 0.646
  Class 1.0  : 0.467
  Class 2.0  : 0.168
  Class 3.0  : 0.428
--------------------
Mean IoU (mIoU) 